# OOP & Design Patterns — Senior Developer Interview Prep

Deep dive into object-oriented Python, magic methods, and design patterns used in production systems.

**Topics Covered:**
1. Magic / Dunder Methods
2. Properties & Descriptors
3. Inheritance, MRO & Mixins
4. Abstract Base Classes
5. Dataclasses & NamedTuples
6. Protocols (Structural Subtyping)
7. Metaclasses
8. Enums
9. Design Patterns
10. SOLID Principles in Python
11. Practice Problems

In [ ]:
import sys
print(f"Python version: {sys.version}")

---
## 1. Magic / Dunder Methods

Dunder (double underscore) methods let your classes integrate with Python's built-in syntax — operators, `len()`, `str()`, iteration, context management, etc.

In [ ]:
# Real-world example: a Money class with operator overloading

class Money:
    """Represents a monetary value with currency."""
    
    def __init__(self, amount, currency="USD"):
        self.amount = round(amount, 2)
        self.currency = currency
    
    def __repr__(self):
        return f"Money({self.amount}, '{self.currency}')"
    
    def __str__(self):
        symbols = {"USD": "$", "EUR": "€", "GBP": "£"}
        sym = symbols.get(self.currency, self.currency)
        return f"{sym}{self.amount:,.2f}"
    
    def _check_currency(self, other):
        if self.currency != other.currency:
            raise ValueError(f"Cannot operate on {self.currency} and {other.currency}")
    
    def __add__(self, other):
        if isinstance(other, Money):
            self._check_currency(other)
            return Money(self.amount + other.amount, self.currency)
        return NotImplemented
    
    def __sub__(self, other):
        if isinstance(other, Money):
            self._check_currency(other)
            return Money(self.amount - other.amount, self.currency)
        return NotImplemented
    
    def __mul__(self, factor):
        if isinstance(factor, (int, float)):
            return Money(self.amount * factor, self.currency)
        return NotImplemented
    
    def __rmul__(self, factor):
        return self.__mul__(factor)
    
    def __eq__(self, other):
        if isinstance(other, Money):
            return self.amount == other.amount and self.currency == other.currency
        return NotImplemented
    
    def __lt__(self, other):
        if isinstance(other, Money):
            self._check_currency(other)
            return self.amount < other.amount
        return NotImplemented
    
    def __le__(self, other):
        return self == other or self < other
    
    def __bool__(self):
        return self.amount != 0
    
    def __hash__(self):
        return hash((self.amount, self.currency))


price = Money(29.99)
tax = Money(2.40)
print(f"Price: {price}")
print(f"Total: {price + tax}")
print(f"Double: {price * 2}")
print(f"3 x price: {3 * price}")  # __rmul__
print(f"repr: {repr(price)}")
print(f"price > tax? {price > tax}")
print(f"bool(Money(0)): {bool(Money(0))}")

In [ ]:
# __contains__, __len__, __getitem__ — make your class act like a collection

class Playlist:
    def __init__(self, name, songs=None):
        self.name = name
        self._songs = list(songs or [])

    def __len__(self):
        return len(self._songs)

    def __getitem__(self, index):
        return self._songs[index]

    def __contains__(self, song):
        return song in self._songs

    def __iter__(self):
        return iter(self._songs)

    def __repr__(self):
        return f"Playlist('{self.name}', {len(self)} songs)"

    def add(self, song):
        self._songs.append(song)

pl = Playlist("Road Trip", ["Bohemian Rhapsody", "Hotel California", "Stairway to Heaven"])

print(f"Length: {len(pl)}")
print(f"First song: {pl[0]}")
print(f"Last two: {pl[-2:]}")
print(f"'Hotel California' in playlist? {'Hotel California' in pl}")

print("\nAll songs:")
for song in pl:
    print(f"  ♫ {song}")

In [ ]:
# __call__ — making instances callable

class Validator:
    """A callable validator that checks multiple rules."""
    def __init__(self):
        self.rules = []
    
    def add_rule(self, rule_fn, message):
        self.rules.append((rule_fn, message))
        return self
    
    def __call__(self, value):
        errors = [msg for rule, msg in self.rules if not rule(value)]
        return errors if errors else None

validate_password = Validator()
validate_password.add_rule(lambda p: len(p) >= 8, "At least 8 characters")
validate_password.add_rule(lambda p: any(c.isupper() for c in p), "At least one uppercase")
validate_password.add_rule(lambda p: any(c.isdigit() for c in p), "At least one digit")

print(validate_password("short"))
print(validate_password("longenough"))
print(validate_password("LongEnough1"))

---
## 2. Properties & Descriptors

Properties provide controlled attribute access. Descriptors are the underlying protocol that powers properties, `classmethod`, `staticmethod`, and more.

In [ ]:
# @property — computed attributes with validation

class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius  # triggers the setter
    
    @property
    def celsius(self):
        return self._celsius
    
    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("Temperature below absolute zero")
        self._celsius = value
    
    @property
    def fahrenheit(self):
        return self._celsius * 9/5 + 32
    
    @fahrenheit.setter
    def fahrenheit(self, value):
        self.celsius = (value - 32) * 5/9

t = Temperature(100)
print(f"{t.celsius}°C = {t.fahrenheit}°F")

t.fahrenheit = 32
print(f"{t.celsius}°C = {t.fahrenheit}°F")

try:
    Temperature(-300)
except ValueError as e:
    print(f"Validation: {e}")

In [ ]:
# Descriptors — the protocol behind @property
# A descriptor is any object that defines __get__, __set__, or __delete__

class TypeChecked:
    """A descriptor that enforces type checking on assignment."""
    def __init__(self, name, expected_type):
        self.name = name
        self.expected_type = expected_type

    def __set_name__(self, owner, name):
        self.storage_name = f"_desc_{name}"

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, self.storage_name, None)

    def __set__(self, obj, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"{self.name} must be {self.expected_type.__name__}, "
                f"got {type(value).__name__}"
            )
        setattr(obj, self.storage_name, value)

class User:
    name = TypeChecked("name", str)
    age = TypeChecked("age", int)
    email = TypeChecked("email", str)
    
    def __init__(self, name, age, email):
        self.name = name
        self.age = age
        self.email = email

user = User("Alice", 30, "alice@example.com")
print(f"{user.name}, age {user.age}")

try:
    user.age = "thirty"  # type error
except TypeError as e:
    print(f"Caught: {e}")

---
## 3. Inheritance, MRO & Mixins

Python uses C3 linearization for Method Resolution Order (MRO). Understanding this is critical for debugging complex class hierarchies.

In [ ]:
# The Diamond Problem & MRO

class A:
    def greet(self):
        return "Hello from A"

class B(A):
    def greet(self):
        return "Hello from B"

class C(A):
    def greet(self):
        return "Hello from C"

class D(B, C):  # diamond: D → B → C → A
    pass

d = D()
print(f"d.greet() = {d.greet()}")  # B wins (listed first in D's bases)
print(f"MRO: {[cls.__name__ for cls in D.__mro__]}")

In [ ]:
# super() — cooperative multiple inheritance

class Base:
    def __init__(self, **kwargs):
        pass  # absorbs remaining kwargs

class LoggingMixin(Base):
    def __init__(self, **kwargs):
        self.log_entries = []
        super().__init__(**kwargs)
    
    def log(self, message):
        from datetime import datetime
        self.log_entries.append(f"{datetime.now():%H:%M:%S} {message}")

class SerializableMixin(Base):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
    
    def to_dict(self):
        return {k: v for k, v in self.__dict__.items() if not k.startswith('_')}

class Service(LoggingMixin, SerializableMixin):
    def __init__(self, name, **kwargs):
        self.name = name
        super().__init__(**kwargs)
    
    def process(self):
        self.log(f"{self.name} processing")
        return "done"

svc = Service("PaymentService")
svc.process()
svc.process()
print(f"Dict: {svc.to_dict()}")
print(f"Logs: {svc.log_entries}")
print(f"MRO: {[c.__name__ for c in Service.__mro__]}")

---
## 4. Abstract Base Classes

ABCs define interfaces that subclasses *must* implement. They prevent instantiation of incomplete implementations.

In [ ]:
from abc import ABC, abstractmethod

class PaymentGateway(ABC):
    @abstractmethod
    def charge(self, amount: float, currency: str) -> str:
        """Process a payment. Return transaction ID."""
        ...
    
    @abstractmethod
    def refund(self, transaction_id: str) -> bool:
        ...
    
    def validate_amount(self, amount):
        """Concrete method — shared logic across all gateways."""
        if amount <= 0:
            raise ValueError("Amount must be positive")

class StripeGateway(PaymentGateway):
    def charge(self, amount, currency="USD"):
        self.validate_amount(amount)
        return f"stripe_txn_{id(self)}_{amount}"
    
    def refund(self, transaction_id):
        print(f"Refunding {transaction_id} via Stripe")
        return True

# Cannot instantiate ABC directly
try:
    gw = PaymentGateway()
except TypeError as e:
    print(f"Cannot instantiate ABC: {e}")

# Incomplete implementation fails too
class BrokenGateway(PaymentGateway):
    def charge(self, amount, currency="USD"):
        return "txn"
    # forgot refund!

try:
    bg = BrokenGateway()
except TypeError as e:
    print(f"Incomplete impl: {e}")

# Complete implementation works
stripe = StripeGateway()
txn = stripe.charge(99.99)
print(f"Transaction: {txn}")

---
## 5. Dataclasses & NamedTuples

Reduce boilerplate for data-holding classes. Know when to use each.

In [ ]:
from dataclasses import dataclass, field, asdict, astuple
from typing import List

@dataclass
class Employee:
    name: str
    department: str
    salary: float
    skills: List[str] = field(default_factory=list)  # mutable default
    _id: int = field(default=0, repr=False)           # excluded from repr
    
    @property
    def annual_salary(self):
        return self.salary * 12
    
    def __post_init__(self):
        """Validation after __init__."""
        if self.salary < 0:
            raise ValueError("Salary cannot be negative")

emp = Employee("Alice", "Engineering", 8000, skills=["Python", "AWS"])
print(f"repr:   {emp}")
print(f"dict:   {asdict(emp)}")
print(f"tuple:  {astuple(emp)}")
print(f"annual: ${emp.annual_salary:,.0f}")

# Automatic __eq__ based on fields
emp2 = Employee("Alice", "Engineering", 8000, skills=["Python", "AWS"])
print(f"\nemp == emp2? {emp == emp2}")

In [ ]:
# Frozen dataclass — immutable (hashable, can be used as dict key)

@dataclass(frozen=True)
class Coordinate:
    lat: float
    lon: float

c1 = Coordinate(37.7749, -122.4194)
c2 = Coordinate(40.7128, -74.0060)

locations = {c1: "San Francisco", c2: "New York"}
print(locations[Coordinate(37.7749, -122.4194)])

try:
    c1.lat = 0  # frozen — can't mutate
except AttributeError as e:
    print(f"Frozen: {e}")

In [ ]:
# Dataclass ordering — __lt__, __le__, __gt__, __ge__

@dataclass(order=True)
class Priority:
    level: int
    name: str = field(compare=False)  # not used in comparisons

tasks = [
    Priority(3, "Low priority task"),
    Priority(1, "Critical bug"),
    Priority(2, "Feature request"),
]

for task in sorted(tasks):
    print(f"  [{task.level}] {task.name}")

In [ ]:
# NamedTuple — immutable, lighter weight, tuple subclass
from typing import NamedTuple

class APIResponse(NamedTuple):
    status: int
    body: str
    headers: dict = {}

resp = APIResponse(200, '{"ok": true}')
print(f"Status: {resp.status}")
print(f"Body: {resp.body}")

# Can unpack like a tuple
status, body, headers = resp
print(f"Unpacked status: {status}")

# NamedTuple vs dataclass:
# - NamedTuple: immutable, tuple subclass, lighter, supports unpacking
# - dataclass: mutable by default, more features, supports inheritance better

---
## 6. Protocols (Structural Subtyping)

Python 3.8+ Protocols enable duck typing with type-checker support — "if it walks like a duck..." but with IDE autocompletion and error checking.

In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Drawable(Protocol):
    def draw(self) -> str: ...
    def area(self) -> float: ...

class Circle:
    def __init__(self, radius):
        self.radius = radius
    def draw(self) -> str:
        return f"Drawing circle r={self.radius}"
    def area(self) -> float:
        return 3.14159 * self.radius ** 2

class Square:
    def __init__(self, side):
        self.side = side
    def draw(self) -> str:
        return f"Drawing square s={self.side}"
    def area(self) -> float:
        return self.side ** 2

def render(shape: Drawable):
    """Works with ANY object that has draw() and area() — no inheritance needed."""
    print(f"{shape.draw()} (area={shape.area():.2f})")

render(Circle(5))
render(Square(4))

print(f"\nCircle satisfies Drawable? {isinstance(Circle(1), Drawable)}")
print(f"str satisfies Drawable? {isinstance('hello', Drawable)}")

---
## 7. Metaclasses

A metaclass is a class whose instances are classes. While rarely needed, understanding them shows deep Python knowledge. `type` is the default metaclass.

In [ ]:
# Everything is an object — classes are instances of type
print(f"type(42)      = {type(42)}")
print(f"type(int)     = {type(int)}")
print(f"type(type)    = {type(type)}")

# Creating a class dynamically with type()
DynamicClass = type("DynamicClass", (object,), {
    "greeting": "Hello",
    "greet": lambda self: f"{self.greeting}, World!"
})

obj = DynamicClass()
print(f"\n{obj.greet()}")
print(f"type: {type(obj)}")

In [ ]:
# Practical metaclass: auto-register all subclasses

class PluginMeta(type):
    registry = {}
    
    def __new__(mcs, name, bases, namespace):
        cls = super().__new__(mcs, name, bases, namespace)
        if bases:  # don't register the base class itself
            PluginMeta.registry[name] = cls
        return cls

class Plugin(metaclass=PluginMeta):
    pass

class JSONPlugin(Plugin):
    def process(self, data):
        return f"Processing JSON: {data}"

class XMLPlugin(Plugin):
    def process(self, data):
        return f"Processing XML: {data}"

class CSVPlugin(Plugin):
    def process(self, data):
        return f"Processing CSV: {data}"

print(f"Registered plugins: {list(PluginMeta.registry.keys())}")

# Instantiate by name
plugin = PluginMeta.registry["JSONPlugin"]()
print(plugin.process('{"key": "value"}'))

In [ ]:
# Modern alternative: __init_subclass__ (Python 3.6+) — often preferred over metaclasses

class Handler:
    _handlers = {}
    
    def __init_subclass__(cls, event_type=None, **kwargs):
        super().__init_subclass__(**kwargs)
        if event_type:
            Handler._handlers[event_type] = cls

class ClickHandler(Handler, event_type="click"):
    def handle(self, data):
        return f"Click: {data}"

class ScrollHandler(Handler, event_type="scroll"):
    def handle(self, data):
        return f"Scroll: {data}"

print(f"Handlers: {Handler._handlers}")

handler = Handler._handlers["click"]()
print(handler.handle("button_submit"))

---
## 8. Enums

Enums provide a set of symbolic names bound to unique values. Better than magic strings/numbers.

In [ ]:
from enum import Enum, auto, IntEnum, Flag

class OrderStatus(Enum):
    PENDING = auto()
    PROCESSING = auto()
    SHIPPED = auto()
    DELIVERED = auto()
    CANCELLED = auto()

    @property
    def is_active(self):
        return self in (OrderStatus.PENDING, OrderStatus.PROCESSING, OrderStatus.SHIPPED)

status = OrderStatus.SHIPPED
print(f"Status: {status}")
print(f"Name: {status.name}, Value: {status.value}")
print(f"Active? {status.is_active}")
print(f"From name: {OrderStatus['PENDING']}")
print(f"From value: {OrderStatus(3)}")

print("\nAll statuses:")
for s in OrderStatus:
    print(f"  {s.name:12} = {s.value} (active={s.is_active})")

In [ ]:
# Flag enum — combinable with bitwise operators

class Permission(Flag):
    READ = auto()
    WRITE = auto()
    EXECUTE = auto()
    ADMIN = READ | WRITE | EXECUTE

user_perms = Permission.READ | Permission.WRITE
print(f"User permissions: {user_perms}")
print(f"Can read? {Permission.READ in user_perms}")
print(f"Can execute? {Permission.EXECUTE in user_perms}")
print(f"Is admin? {user_perms == Permission.ADMIN}")

---
## 9. Design Patterns

Pythonic implementations of the most commonly asked design patterns.

In [ ]:
# STRATEGY PATTERN — swap algorithms at runtime
# Real-world: different pricing strategies for an e-commerce site

from dataclasses import dataclass
from typing import Callable

def regular_pricing(price: float) -> float:
    return price

def member_pricing(price: float) -> float:
    return price * 0.9  # 10% discount

def vip_pricing(price: float) -> float:
    return price * 0.75  # 25% discount

@dataclass
class ShoppingCart:
    items: list = field(default_factory=list)
    pricing_strategy: Callable = regular_pricing
    
    def add(self, item: str, price: float):
        self.items.append((item, price))
    
    def total(self) -> float:
        return sum(self.pricing_strategy(price) for _, price in self.items)

cart = ShoppingCart(pricing_strategy=vip_pricing)
cart.add("Laptop", 1000)
cart.add("Mouse", 50)
print(f"VIP total: ${cart.total():,.2f}")

cart.pricing_strategy = regular_pricing
print(f"Regular total: ${cart.total():,.2f}")

In [ ]:
# OBSERVER PATTERN — event system / pub-sub
# Real-world: notifying multiple systems when an order is placed

from collections import defaultdict

class EventBus:
    def __init__(self):
        self._subscribers = defaultdict(list)
    
    def subscribe(self, event_type, callback):
        self._subscribers[event_type].append(callback)
    
    def publish(self, event_type, data=None):
        for callback in self._subscribers[event_type]:
            callback(data)

bus = EventBus()

# Different systems subscribe to "order_placed"
bus.subscribe("order_placed", lambda order: print(f"  📧 Email: Order {order['id']} confirmed"))
bus.subscribe("order_placed", lambda order: print(f"  📦 Warehouse: Ship order {order['id']}"))
bus.subscribe("order_placed", lambda order: print(f"  📊 Analytics: Track order {order['id']}"))

print("Publishing order_placed event:")
bus.publish("order_placed", {"id": "ORD-001", "total": 99.99})

In [ ]:
# FACTORY PATTERN — create objects without specifying exact class
# Real-world: creating different notification channels

from abc import ABC, abstractmethod

class Notifier(ABC):
    @abstractmethod
    def send(self, message: str) -> str: ...

class EmailNotifier(Notifier):
    def send(self, message):
        return f"Email sent: {message}"

class SMSNotifier(Notifier):
    def send(self, message):
        return f"SMS sent: {message}"

class SlackNotifier(Notifier):
    def send(self, message):
        return f"Slack sent: {message}"

class NotifierFactory:
    _notifiers = {
        "email": EmailNotifier,
        "sms": SMSNotifier,
        "slack": SlackNotifier,
    }
    
    @classmethod
    def create(cls, channel: str) -> Notifier:
        notifier_cls = cls._notifiers.get(channel)
        if not notifier_cls:
            raise ValueError(f"Unknown channel: {channel}")
        return notifier_cls()
    
    @classmethod
    def register(cls, channel: str, notifier_cls):
        cls._notifiers[channel] = notifier_cls

for channel in ["email", "sms", "slack"]:
    notifier = NotifierFactory.create(channel)
    print(notifier.send("Your order has shipped!"))

In [ ]:
# DECORATOR PATTERN (structural, not Python decorator syntax)
# Real-world: adding features to a data pipeline

class DataSource:
    def read(self):
        return "raw_data_from_db"

class DataSourceDecorator:
    def __init__(self, source):
        self._source = source
    
    def read(self):
        return self._source.read()

class CachingDecorator(DataSourceDecorator):
    def __init__(self, source):
        super().__init__(source)
        self._cache = None
    
    def read(self):
        if self._cache is None:
            print("  Cache MISS — fetching from source")
            self._cache = super().read()
        else:
            print("  Cache HIT")
        return self._cache

class LoggingDecorator(DataSourceDecorator):
    def read(self):
        print("  [LOG] Reading data...")
        data = super().read()
        print(f"  [LOG] Got {len(data)} chars")
        return data

# Compose: logging → caching → source
source = LoggingDecorator(CachingDecorator(DataSource()))
print("First read:")
source.read()
print("\nSecond read:")
source.read()

In [ ]:
# CHAIN OF RESPONSIBILITY — pass request through a chain of handlers
# Real-world: middleware pipeline (like Django/Flask/Express)

class Middleware:
    def __init__(self, next_handler=None):
        self.next = next_handler
    
    def handle(self, request):
        if self.next:
            return self.next.handle(request)
        return request

class AuthMiddleware(Middleware):
    def handle(self, request):
        if not request.get("token"):
            return {"error": "Unauthorized", "status": 401}
        request["user"] = "authenticated_user"
        print("  ✓ Auth passed")
        return super().handle(request)

class RateLimitMiddleware(Middleware):
    def handle(self, request):
        print("  ✓ Rate limit OK")
        return super().handle(request)

class LoggingMiddleware(Middleware):
    def handle(self, request):
        print(f"  ✓ Logged: {request.get('path', '/')}")
        return super().handle(request)

# Build chain: Logging → RateLimit → Auth → Handler
pipeline = LoggingMiddleware(RateLimitMiddleware(AuthMiddleware()))

print("Valid request:")
print(pipeline.handle({"path": "/api/data", "token": "abc123"}))

print("\nNo token:")
print(pipeline.handle({"path": "/api/data"}))

---
## 10. SOLID Principles in Python

Quick examples of each principle applied in Python.

In [ ]:
# S — Single Responsibility Principle
# Each class should have one reason to change.

# BAD: one class doing everything
class UserManagerBad:
    def create_user(self, data): ...
    def send_welcome_email(self, user): ...  # email concern
    def generate_report(self, users): ...     # reporting concern

# GOOD: separate responsibilities
class UserRepository:
    def create(self, data): ...
    def find(self, user_id): ...

class EmailService:
    def send_welcome(self, user): ...

class UserReportGenerator:
    def generate(self, users): ...

print("SRP: Split UserManager into UserRepository + EmailService + ReportGenerator")

In [ ]:
# O — Open/Closed Principle
# Open for extension, closed for modification.

# BAD: modifying existing code to add new shapes
class AreaCalculatorBad:
    def calculate(self, shape):
        if shape["type"] == "circle":
            return 3.14 * shape["radius"] ** 2
        elif shape["type"] == "rectangle":
            return shape["width"] * shape["height"]
        # must modify this class for every new shape!

# GOOD: extend via new classes
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    @abstractmethod
    def area(self) -> float: ...

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius
    def area(self):
        return math.pi * self.radius ** 2

class Rectangle(Shape):
    def __init__(self, w, h):
        self.w, self.h = w, h
    def area(self):
        return self.w * self.h

# Adding Triangle doesn't touch existing code
class Triangle(Shape):
    def __init__(self, base, height):
        self.base, self.height = base, height
    def area(self):
        return 0.5 * self.base * self.height

shapes = [Circle(5), Rectangle(4, 6), Triangle(3, 8)]
for s in shapes:
    print(f"{s.__class__.__name__}: area = {s.area():.2f}")

In [ ]:
# D — Dependency Inversion Principle
# Depend on abstractions, not concretions.

from abc import ABC, abstractmethod

class MessageSender(ABC):
    @abstractmethod
    def send(self, to: str, body: str): ...

class EmailSender(MessageSender):
    def send(self, to, body):
        print(f"Email to {to}: {body}")

class SMSSender(MessageSender):
    def send(self, to, body):
        print(f"SMS to {to}: {body}")

class NotificationService:
    def __init__(self, sender: MessageSender):  # depends on abstraction
        self.sender = sender
    
    def notify(self, user, message):
        self.sender.send(user, message)

# Easy to swap implementations
service = NotificationService(EmailSender())
service.notify("alice@example.com", "Your order shipped!")

service = NotificationService(SMSSender())
service.notify("+1234567890", "Your order shipped!")

---
## 11. Practice Problems

### Problem 1: Implement a `LinkedList` with dunder methods

Create a linked list that supports:
- `len(ll)` — returns length
- `ll[i]` — index access
- `for item in ll` — iteration
- `item in ll` — membership test
- `str(ll)` — readable string like `"1 -> 2 -> 3 -> None"`
- `ll1 + ll2` — concatenation

In [ ]:
# YOUR SOLUTION HERE


In [ ]:
# SOLUTION
class Node:
    __slots__ = ('value', 'next')
    def __init__(self, value, next_node=None):
        self.value = value
        self.next = next_node

class LinkedList:
    def __init__(self, items=None):
        self.head = None
        self._length = 0
        if items:
            for item in items:
                self.append(item)
    
    def append(self, value):
        new_node = Node(value)
        if not self.head:
            self.head = new_node
        else:
            current = self.head
            while current.next:
                current = current.next
            current.next = new_node
        self._length += 1
    
    def __len__(self):
        return self._length
    
    def __getitem__(self, index):
        if index < 0:
            index = self._length + index
        if index < 0 or index >= self._length:
            raise IndexError("list index out of range")
        current = self.head
        for _ in range(index):
            current = current.next
        return current.value
    
    def __iter__(self):
        current = self.head
        while current:
            yield current.value
            current = current.next
    
    def __contains__(self, value):
        return any(item == value for item in self)
    
    def __str__(self):
        return " -> ".join(str(item) for item in self) + " -> None"
    
    def __repr__(self):
        return f"LinkedList({list(self)})"
    
    def __add__(self, other):
        if isinstance(other, LinkedList):
            new_list = LinkedList(self)
            for item in other:
                new_list.append(item)
            return new_list
        return NotImplemented

ll = LinkedList([1, 2, 3, 4, 5])
print(f"str:      {ll}")
print(f"len:      {len(ll)}")
print(f"ll[2]:    {ll[2]}")
print(f"ll[-1]:   {ll[-1]}")
print(f"3 in ll:  {3 in ll}")
print(f"concat:   {ll + LinkedList([6, 7, 8])}")

### Problem 2: Implement the Builder pattern

Create a `QueryBuilder` that builds SQL queries fluently:

```python
query = (QueryBuilder()
    .select("name", "email")
    .from_table("users")
    .where("age > 18")
    .where("active = true")
    .order_by("name")
    .limit(10)
    .build())
# "SELECT name, email FROM users WHERE age > 18 AND active = true ORDER BY name LIMIT 10"
```

In [ ]:
# YOUR SOLUTION HERE


In [ ]:
# SOLUTION
class QueryBuilder:
    def __init__(self):
        self._select = []
        self._table = None
        self._where = []
        self._order_by = []
        self._limit = None
    
    def select(self, *columns):
        self._select.extend(columns)
        return self  # enable chaining
    
    def from_table(self, table):
        self._table = table
        return self
    
    def where(self, condition):
        self._where.append(condition)
        return self
    
    def order_by(self, *columns):
        self._order_by.extend(columns)
        return self
    
    def limit(self, n):
        self._limit = n
        return self
    
    def build(self):
        if not self._table:
            raise ValueError("Table is required")
        
        parts = []
        cols = ", ".join(self._select) if self._select else "*"
        parts.append(f"SELECT {cols}")
        parts.append(f"FROM {self._table}")
        
        if self._where:
            parts.append(f"WHERE {' AND '.join(self._where)}")
        if self._order_by:
            parts.append(f"ORDER BY {', '.join(self._order_by)}")
        if self._limit is not None:
            parts.append(f"LIMIT {self._limit}")
        
        return " ".join(parts)

query = (QueryBuilder()
    .select("name", "email")
    .from_table("users")
    .where("age > 18")
    .where("active = true")
    .order_by("name")
    .limit(10)
    .build())

print(query)

### Problem 3: Implement a `Registry` using `__init_subclass__`

Create a `Serializer` base class where subclasses auto-register by file extension:

```python
class JSONSerializer(Serializer, ext=".json"): ...
class YAMLSerializer(Serializer, ext=".yaml"): ...

serializer = Serializer.for_file("config.json")  # returns JSONSerializer instance
```

In [ ]:
# YOUR SOLUTION HERE


In [ ]:
# SOLUTION
import os

class Serializer:
    _registry = {}

    def __init_subclass__(cls, ext=None, **kwargs):
        super().__init_subclass__(**kwargs)
        if ext:
            if isinstance(ext, str):
                ext = [ext]
            for e in ext:
                Serializer._registry[e] = cls

    @classmethod
    def for_file(cls, filename):
        _, ext = os.path.splitext(filename)
        serializer_cls = cls._registry.get(ext)
        if not serializer_cls:
            raise ValueError(f"No serializer for {ext}")
        return serializer_cls()

class JSONSerializer(Serializer, ext=".json"):
    def serialize(self, data):
        return f"JSON: {data}"

class YAMLSerializer(Serializer, ext=[".yaml", ".yml"]):
    def serialize(self, data):
        return f"YAML: {data}"

class TOMLSerializer(Serializer, ext=".toml"):
    def serialize(self, data):
        return f"TOML: {data}"

print(f"Registry: {Serializer._registry}")

for filename in ["config.json", "settings.yaml", "pyproject.toml"]:
    s = Serializer.for_file(filename)
    print(f"{filename} → {s.__class__.__name__}: {s.serialize({'key': 'value'})}")

### Problem 4: What does this code print? (MRO + super)

Predict the output:

```python
class A:
    def method(self):
        print("A", end=" ")

class B(A):
    def method(self):
        print("B", end=" ")
        super().method()

class C(A):
    def method(self):
        print("C", end=" ")
        super().method()

class D(B, C):
    def method(self):
        print("D", end=" ")
        super().method()

D().method()
```

In [ ]:
# Think about the MRO, then run to verify

class A:
    def method(self):
        print("A", end=" ")

class B(A):
    def method(self):
        print("B", end=" ")
        super().method()

class C(A):
    def method(self):
        print("C", end=" ")
        super().method()

class D(B, C):
    def method(self):
        print("D", end=" ")
        super().method()

D().method()
print()

# MRO is D → B → C → A → object
# D.method: prints "D", calls super() → B.method
# B.method: prints "B", calls super() → C.method (NOT A!)
# C.method: prints "C", calls super() → A.method
# A.method: prints "A"
# Output: D B C A
print(f"MRO: {[c.__name__ for c in D.__mro__]}")